# 1) Load data

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os

In [ ]:
#Path to workbook
excel_path = "data/responses/Evaluation Response Analysis.xlsx"

#Load sheet
df = pd.read_excel(excel_path, sheet_name="binomial_analysis")

df.head()

,participant_id,participant_persona,source_clip_id,better_model_overall,more_carnatic_model,better_continuity_model,performer_type
0,PA00930C,Knows Carnatic Music,Karpagambike_seg0044,finetuned,finetuned,finetuned,Vocal
1,PA00930C,Knows Carnatic Music,Ramanatham_Bhajeham_seg0029,baseline,baseline,finetuned,Vocal
2,PA00930C,Knows Carnatic Music,Raksha_Bettare_seg0201,finetuned,finetuned,baseline,Vocal
3,PA00930C,Knows Carnatic Music,Deva_Deva_Jagadeesha_seg0006,finetuned,finetuned,finetuned,Instrumental - Violin
4,PA00930C,Knows Carnatic Music,Seshachala_Nayakam_seg0142,finetuned,finetuned,finetuned,Instrumental - Violin


# 2) Clean and standardize values

In [3]:
#Standardize column names
df.columns = [c.strip() for c in df.columns]

#Standardize text values
for col in [
                "participant_persona",
                "better_model_overall",
                "more_carnatic_model",
                "better_continuity_model",
                "performer_type"
            ]:
    df[col] = df[col].astype(str).str.strip().str.lower()

#Optional: inspect unique values
for col in [
                "participant_persona",
                "better_model_overall",
                "more_carnatic_model",
                "better_continuity_model",
                "performer_type"
            ]:
    print(f"{col}: {df[col].unique()}")

participant_persona: ['knows carnatic music' 'doesn’t know carnatic music']
better_model_overall: ['finetuned' 'baseline']
more_carnatic_model: ['finetuned' 'baseline']
better_continuity_model: ['finetuned' 'baseline']
performer_type: ['vocal' 'instrumental - violin' 'percussion - mirudhangam']


In [4]:
len(df)

192

# 3) Main binomial-test analysis

In [5]:
outcome_cols = {
                    "better_model_overall": "Better Musicality",
                    "more_carnatic_model": "Better Carnatic Authenticity",
                    "better_continuity_model": "Better Carnatic Continuity"
                }

results = []

for col, label in outcome_cols.items():
    sub = df[df[col].isin(["baseline", "finetuned"])].copy()


    n_total = len(sub)
    n_finetuned = (sub[col] == "finetuned").sum()
    n_baseline = (sub[col] == "baseline").sum()

    # Two-sided binomial test against 0.5
    test = binomtest(k=n_finetuned, n=n_total, p=0.5, alternative="greater")

    results.append({
                        "outcome": label,
                        "n_total": n_total,
                        "baseline_count": n_baseline,
                        "finetuned_count": n_finetuned,
                        "finetuned_percent": 100 * n_finetuned / n_total if n_total > 0 else np.nan,
                        "raw_p_value": test.pvalue
                    })

results_df = pd.DataFrame(results)
results_df

,outcome,n_total,baseline_count,finetuned_count,finetuned_percent,raw_p_value
0,Better Musicality,192,71,121,63.020833,0.000189
1,Better Carnatic Authenticity,192,68,124,64.583333,0.000032
2,Better Carnatic Continuity,192,72,120,62.500000,0.000328


# 4) Holm correction

In [6]:
reject, pvals_holm, _, _ = multipletests(
                                            results_df["raw_p_value"],
                                            alpha=0.05,
                                            method="holm"
                                        )

results_df["holm_corrected_p"] = pvals_holm
results_df["significant_after_holm"] = reject

results_df = results_df.round({
                                "finetuned_percent": 2,
                                "raw_p_value": 6,
                                "holm_corrected_p": 6
                            })

results_df

,outcome,n_total,baseline_count,finetuned_count,finetuned_percent,raw_p_value,holm_corrected_p,significant_after_holm
0,Better Musicality,192,71,121,63.02,0.000189,0.000379,True
1,Better Carnatic Authenticity,192,68,124,64.58,0.000032,0.000097,True
2,Better Carnatic Continuity,192,72,120,62.50,0.000328,0.000379,True


# 5) Save the results to an Excel file

In [7]:
with pd.ExcelWriter(excel_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    results_df.to_excel(writer, sheet_name='binomial_analysis_results', index=False)

print("Saved results to Excel sheet: binomial_analysis_results")

Saved results to Excel sheet: binomial_analysis_results


# 6) subgroup analysis

## By participant persona

In [12]:
subgroup_rows = []

for persona in sorted(df["participant_persona"].dropna().unique()):
    df_persona = df[df["participant_persona"] == persona].copy()

    for col, label in outcome_cols.items():
        sub = df_persona[df_persona[col].isin(["baseline", "finetuned"])].copy()

        n_total = len(sub)
        if n_total == 0:
            continue

        n_finetuned = (sub[col] == "finetuned").sum()
        n_baseline = (sub[col] == "baseline").sum()

        test = binomtest(k=n_finetuned, n=n_total, p=0.5, alternative="greater")

        subgroup_rows.append({
            "group_type": "participant_persona",
            "group_value": persona,
            "outcome": label,
            "n_total": n_total,
            "baseline_count": n_baseline,
            "finetuned_count": n_finetuned,
            "finetuned_percent": 100 * n_finetuned / n_total,
            "raw_p_value": test.pvalue
        })

persona_results_df = pd.DataFrame(subgroup_rows)
persona_results_df.head()

,group_type,group_value,outcome,n_total,baseline_count,finetuned_count,finetuned_percent,raw_p_value
0,participant_persona,doesn’t know carnatic music,Better Musicality,102,43,59,57.843137,0.068549
1,participant_persona,doesn’t know carnatic music,Better Carnatic Authenticity,102,43,59,57.843137,0.068549
2,participant_persona,doesn’t know carnatic music,Better Carnatic Continuity,102,44,58,56.862745,0.098895
3,participant_persona,knows carnatic music,Better Musicality,90,28,62,68.888889,0.000219
4,participant_persona,knows carnatic music,Better Carnatic Authenticity,90,25,65,72.222222,0.000015


In [13]:
persona_corrected = []

for persona in persona_results_df["group_value"].unique():
    tmp = persona_results_df[persona_results_df["group_value"] == persona].copy()

    reject, pvals_holm, _, _ = multipletests(
        tmp["raw_p_value"],
        alpha=0.05,
        method="holm"
    )

    tmp["holm_corrected_p"] = pvals_holm
    tmp["significant_after_holm"] = reject
    persona_corrected.append(tmp)

persona_results_df = pd.concat(persona_corrected, ignore_index=True)
persona_results_df = persona_results_df.round({
    "finetuned_percent": 2,
    "raw_p_value": 6,
    "holm_corrected_p": 6
})

persona_results_df

,group_type,group_value,outcome,n_total,baseline_count,finetuned_count,finetuned_percent,raw_p_value,holm_corrected_p,significant_after_holm
0,participant_persona,doesn’t know carnatic music,Better Musicality,102,43,59,57.84,0.068549,0.205648,False
1,participant_persona,doesn’t know carnatic music,Better Carnatic Authenticity,102,43,59,57.84,0.068549,0.205648,False
2,participant_persona,doesn’t know carnatic music,Better Carnatic Continuity,102,44,58,56.86,0.098895,0.205648,False
3,participant_persona,knows carnatic music,Better Musicality,90,28,62,68.89,0.000219,0.000438,True
4,participant_persona,knows carnatic music,Better Carnatic Authenticity,90,25,65,72.22,0.000015,0.000044,True
5,participant_persona,knows carnatic music,Better Carnatic Continuity,90,28,62,68.89,0.000219,0.000438,True


## By performer type 

In [14]:
subgroup_rows = []

for performer in sorted(df["performer_type"].dropna().unique()):
    df_perf = df[df["performer_type"] == performer].copy()

    for col, label in outcome_cols.items():
        sub = df_perf[df_perf[col].isin(["baseline", "finetuned"])].copy()

        n_total = len(sub)
        if n_total == 0:
            continue

        n_finetuned = (sub[col] == "finetuned").sum()
        n_baseline = (sub[col] == "baseline").sum()

        test = binomtest(k=n_finetuned, n=n_total, p=0.5, alternative="two-sided")

        subgroup_rows.append({
            "group_type": "performer_type",
            "group_value": performer,
            "outcome": label,
            "n_total": n_total,
            "baseline_count": n_baseline,
            "finetuned_count": n_finetuned,
            "finetuned_percent": 100 * n_finetuned / n_total,
            "raw_p_value": test.pvalue
        })

performer_results_df = pd.DataFrame(subgroup_rows)
performer_results_df.head()

,group_type,group_value,outcome,n_total,baseline_count,finetuned_count,finetuned_percent,raw_p_value
0,performer_type,instrumental - violin,Better Musicality,64,11,53,82.8125,1.005858e-07
1,performer_type,instrumental - violin,Better Carnatic Authenticity,64,11,53,82.8125,1.005858e-07
2,performer_type,instrumental - violin,Better Carnatic Continuity,64,14,50,78.1250,7.069488e-06
3,performer_type,percussion - mirudhangam,Better Musicality,32,10,22,68.7500,5.010246e-02
4,performer_type,percussion - mirudhangam,Better Carnatic Authenticity,32,10,22,68.7500,5.010246e-02


In [ ]:
performer_corrected = []

for performer in performer_results_df["group_value"].unique():
    tmp = performer_results_df[performer_results_df["group_value"] == performer].copy()

    reject, pvals_holm, _, _ = multipletests(
        tmp["raw_p_value"],
        alpha=0.05,
        method="holm"
    )

    tmp["holm_corrected_p"] = pvals_holm
    tmp["significant_after_holm"] = reject
    performer_corrected.append(tmp)

performer_results_df = pd.concat(performer_corrected, ignore_index=True)
performer_results_df = performer_results_df.round({
    "finetuned_percent": 2,
    "raw_p_value": 6,
    "holm_corrected_p": 6
})

performer_results_df

,group_type,group_value,outcome,n_total,baseline_count,finetuned_count,finetuned_percent,raw_p_value,holm_corrected_p,significant_after_holm
0,performer_type,instrumental - violin,Better Musicality,64,11,53,82.81,0.000000,0.000000,True
1,performer_type,instrumental - violin,Better Carnatic Authenticity,64,11,53,82.81,0.000000,0.000000,True
2,performer_type,instrumental - violin,Better Carnatic Continuity,64,14,50,78.12,0.000007,0.000007,True
3,performer_type,percussion - mirudhangam,Better Musicality,32,10,22,68.75,0.050102,0.100205,False
4,performer_type,percussion - mirudhangam,Better Carnatic Authenticity,32,10,22,68.75,0.050102,0.100205,False
5,performer_type,percussion - mirudhangam,Better Carnatic Continuity,32,9,23,71.88,0.020062,0.060185,False
6,performer_type,vocal,Better Musicality,96,50,46,47.92,0.759649,1.000000,False
7,performer_type,vocal,Better Carnatic Authenticity,96,47,49,51.04,0.918778,1.000000,False
8,performer_type,vocal,Better Carnatic Continuity,96,49,47,48.96,0.918778,1.000000,False
